# Bases de données — le cours à lire avant les exercices

Ce cours se lit **avant** d'attaquer les exercices du polycopié. Il part de zéro : il suffit de savoir ce qu'est un tableau de données (un fichier CSV, vu en Première).

Il suit le **programme officiel** de Terminale NSI (thème « Bases de données », publié par le ministère de l'Éducation nationale), dont voici les capacités attendues :

| Notion du programme | Ce qu'il faut savoir faire | Dans ce cours |
|:---|:---|:---|
| Modèle relationnel : relation, attribut, domaine, clef primaire, clef étrangère, schéma relationnel | identifier ces notions | parties 1, 9 et 10 |
| Base de données relationnelle | distinguer la **structure** d'une base de son **contenu** ; repérer des anomalies (redondances) dans un schéma | parties 1 et 11 |
| Système de gestion de bases de données (SGBD) | connaître ses services : persistance, accès concurrents, efficacité, sécurité | partie 2 |
| Langage SQL | construire des requêtes avec `SELECT`, `FROM`, `WHERE`, `JOIN` ; `INSERT`, `UPDATE`, `DELETE` ; on peut utiliser `DISTINCT`, `ORDER BY` et les fonctions d'agrégation, **sans** `GROUP BY` ni `HAVING` | parties 4 à 8, 10 et 12 |

**Comment lire ce cours :** chaque notion est expliquée, puis montrée sur un **exemple tapé dans la console H2** (le logiciel utilisé en classe). Le résultat affiché sous chaque exemple est celui qu'a réellement donné H2. Le mieux est de retaper les exemples soi-même.

**Plan :**
1. Le vocabulaire : relation, attribut, enregistrement, domaine
2. Le SGBD et la console H2
3. Les types (domaines de valeurs)
4. Créer une table : `CREATE TABLE`
5. Remplir une table : `INSERT INTO`, et `RUNSCRIPT`
6. Les outils Python de l'exercice 1 : lire un CSV, écrire un fichier
7. Interroger une table : `SELECT … FROM … WHERE …`
8. Compter, moyenne, minimum, maximum
9. Clé primaire
10. Plusieurs tables : clé étrangère et jointure `JOIN`
11. Pourquoi plusieurs tables ? (les anomalies)
12. Modifier et supprimer : `UPDATE`, `DELETE`
13. La méthode pour écrire une requête, et la fiche récapitulative

---
# 1. Le vocabulaire

Une **base de données relationnelle** range les données dans des **tableaux**. Exemple : un tableau des pays.

| id | code | name | continent |
|:---|:---|:---|:---|
| 1 | FR | France | EU |
| 2 | ES | Spain | EU |
| 3 | BR | Brazil | SA |

Les mots à connaître (le cours du polycopié emploie les deux colonnes) :

| Mot du cours | Mot de tous les jours | Dans l'exemple |
|:---|:---|:---|
| **relation** | table, tableau | la table `Countries` |
| **attribut** | colonne | `id`, `code`, `name`, `continent` |
| **enregistrement** (ou n-uplet) | ligne | `1, FR, France, EU` |
| **domaine** d'un attribut | les valeurs autorisées, son **type** | `id` : un entier ; `name` : du texte |

- Chaque **relation** décrit une sorte d'objets du monde réel (des pays, des aéroports, des élèves…). Chaque **ligne** est un objet ; chaque **colonne** est une caractéristique de ces objets.
- **Structure et contenu** : la **structure** d'une table, c'est la liste de ses colonnes et de leurs types (elle change rarement) ; le **contenu**, ce sont les lignes (elles changent tout le temps). On décrit la structure par un **schéma** :

  Countries(id : INT, code : CHAR(2), name : VARCHAR, continent : VARCHAR)

- **Pourquoi pas un simple fichier CSV ?** Un CSV convient pour quelques centaines de lignes lues par un seul programme. Une base de données gère des millions de lignes, utilisées **en même temps** par beaucoup d'utilisateurs, en **vérifiant** que les données restent cohérentes. C'est le rôle du SGBD.

---
# 2. Le SGBD et la console H2

Un **SGBD** (système de gestion de bases de données) est le logiciel qui stocke la base et répond aux demandes. En classe, on utilise le SGBD **H2**. D'autres exemples : SQLite, MySQL, PostgreSQL, Oracle.

**Les services rendus par un SGBD** (à savoir citer) :
- **persistance** : les données sont enregistrées sur le disque, elles restent quand on éteint l'ordinateur ;
- **gestion des accès concurrents** : plusieurs utilisateurs peuvent lire et modifier la base **en même temps** sans tout mélanger (deux personnes ne peuvent pas réserver le même siège) ;
- **efficacité** : il trouve très vite une ligne parmi des millions ;
- **sécurisation des accès** : chaque utilisateur a un mot de passe et des droits (lire seulement, ou modifier) ;
- en plus, il **vérifie la cohérence** des données (les *contraintes d'intégrité*, parties 3, 9 et 10).

**On parle au SGBD avec le langage SQL.** Chaque demande s'appelle une **requête** (ou instruction) et se termine par un **point-virgule** `;`. Les majuscules aux mots-clés (`SELECT`, `FROM`…) sont une habitude de lecture, pas une obligation. Les retours à la ligne et les espaces n'ont pas d'importance.

**Utiliser la console H2 :**
1. lancer « H2 Console » : une page s'ouvre dans le navigateur ;
2. se connecter : JDBC URL `jdbc:h2:~/nsi` (le fichier de la base, dans le dossier personnel), utilisateur `sa`, mot de passe vide, bouton **Connecter** ;
3. taper la requête dans la grande zone de texte, puis cliquer sur **Exécuter** (ou Ctrl + Entrée) ;
4. le résultat s'affiche en dessous, sous forme de tableau.

On peut taper plusieurs requêtes d'un coup (chacune finie par `;`) : H2 les exécute dans l'ordre.

> Pour repartir d'une base vide : `DROP ALL OBJECTS;` efface toutes les tables de la base connectée.

---
# 3. Les types (domaines de valeurs)

Chaque colonne a un **type** : le SGBD **refuse** une valeur qui n'est pas du bon type. C'est la **contrainte de domaine**.

| Type | Contenu | Exemple de valeur en SQL |
|:---|:---|:---|
| `INT` | nombre entier | `302687` |
| `REAL` (ou `FLOAT`) | nombre à virgule (avec un **point**) | `48.72` |
| `CHAR(n)` | texte d'**au plus** n caractères | `'FR'` pour `CHAR(2)` |
| `VARCHAR` | texte de longueur quelconque | `'France'` |
| `DATE` | une date | `'2026-11-14'` (année-mois-jour) |
| `TIMESTAMP` | une date et une heure | `'2026-11-14 08:30:00'` |
| `BOOLEAN` | vrai ou faux | `TRUE`, `FALSE` |

⚠️ **Le texte s'écrit entre apostrophes** `'...'` (pas entre guillemets). Tout ce qui est entre les apostrophes compte, **espaces compris** : `' FR '` fait 4 caractères. Les nombres s'écrivent **sans** apostrophes.

---
# 4. Créer une table : `CREATE TABLE`

On décrit la **structure** : le nom de la table, puis chaque colonne avec son type, séparées par des virgules (pas de virgule après la dernière).

*Dans la console H2 :*

```sql
CREATE TABLE Countries (
    id INT,
    code CHAR(2),
    name VARCHAR,
    continent VARCHAR );
```

*Résultat dans la console H2 :*

```text
(Update count: 0)
```

- `Update count: 0` : la table est créée, elle est **vide** (aucune ligne modifiée).
- Si on l'exécute une deuxième fois, H2 répond que la table existe déjà. Pour la supprimer (structure **et** contenu) : `DROP TABLE Countries;`.
- C'est exactement la question 1 de l'exercice 1 du polycopié.

---
# 5. Remplir une table : `INSERT INTO`

Une requête `INSERT INTO` ajoute **une ligne**. On donne les colonnes, puis les valeurs **dans le même ordre**.

*Dans la console H2 :*

```sql
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME   | CONTINENT
1  | FR   | France | EU
2  | ES   | Spain  | EU
3  | BR   | Brazil | SA
(3 rows)
```

- `Update count: 1` : une ligne ajoutée. `SELECT * FROM Countries;` affiche toute la table (partie 7).
- Si on donne une valeur pour **toutes** les colonnes, dans l'ordre de la table, on peut omettre la liste des colonnes : `INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');`.

**Le SGBD vérifie les types.** Une chaîne de 4 caractères dans une colonne `CHAR(2)` est refusée :

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU');
```

*Résultat dans la console H2 :*

```text
Erreur : Valeur trop longue pour la colonne "CODE CHARACTER(2)": "' IM ' (4)"
Value too long for column "CODE CHARACTER(2)": "' IM ' (4)"; SQL statement:
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU') [22001-252]
```

⚠️ C'est **le piège du copier-coller depuis le PDF** du polycopié : il ajoute des espaces autour des apostrophes. Il faut taper `'IM'`, collé aux apostrophes.

**Une apostrophe dans un texte** (*Côte d'Ivoire*) : comme l'apostrophe termine le texte, on la **double** : `'Côte d''Ivoire'`.

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
(Update count: 1)

INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
(Update count: 1)

INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME          | CONTINENT
1  | FR   | France        | EU
2  | ES   | Spain         | EU
3  | BR   | Brazil        | SA
4  | IM   | Isle of Man   | EU
5  | CI   | Côte d'Ivoire | AF
6  | JP   | Japan         | AS
(6 rows)
```

### Exécuter un fichier : `RUNSCRIPT`

Un fichier `.sql` est un simple **fichier texte** qui contient des requêtes, l'une après l'autre. Au lieu de les recopier, on demande à H2 de toutes les exécuter :

```sql
RUNSCRIPT FROM 'C:/Users/Iheb/Downloads/countries.sql';
```

(entre apostrophes, le chemin complet du fichier, avec des `/` même sous Windows ; sur Mac : `'/Users/iheb/Downloads/countries.sql'`). C'est ainsi qu'on charge les données des exercices 2 (aéroports) et 3 (Vélib').

---
# 6. Les outils Python de l'exercice 1

La question 2 (b) de l'exercice 1 demande un programme Python qui **fabrique** le fichier `countries.sql` à partir du fichier `countries.csv`. Il faut trois outils.

### Outil 1 : écrire dans un fichier texte

```python
fichier = open("essai.txt", "w", encoding="utf-8")   # "w" = write : on crée le fichier (ou on l'efface)
fichier.write("première ligne\n")                      # "\n" = passer à la ligne
fichier.write("deuxième ligne\n")
fichier.close()                                         # obligatoire : c'est là que tout est enregistré
```
`write` écrit **exactement** la chaîne donnée : sans `"\n"`, tout serait sur la même ligne.

### Outil 2 : lire un fichier CSV avec le module `csv`

Un fichier CSV contient une ligne d'en-tête (les noms des colonnes), puis une ligne par enregistrement :
```text
"id","code","name","continent"
302672,"AD","Andorra","EU"
302618,"AE","United Arab Emirates","AS"
```
`csv.DictReader` lit le fichier et donne chaque ligne sous la forme d'un **dictionnaire** dont les clés sont les noms des colonnes :
```python
import csv

fichier = open("countries.csv", encoding="utf-8")
for pays in csv.DictReader(fichier):
    print(pays["name"], pays["code"])      # pays vaut {"id": "302672", "code": "AD", "name": "Andorra", "continent": "EU"}
fichier.close()
```
⚠️ Toutes les valeurs lues sont des **chaînes** : `pays["id"]` vaut `"302672"`, pas le nombre.

### Outil 3 : fabriquer une requête SQL avec `+`

On colle des morceaux de texte avec `+` :
```python
ligne = "INSERT INTO Countries VALUES (" + pays["id"] + ", '" + pays["code"] + "', '" + pays["name"] + "', '" + pays["continent"] + "');"
```
Pour Andorre, `ligne` vaut : `INSERT INTO Countries VALUES (302672, 'AD', 'Andorra', 'EU');`

Regarder les guillemets de près : la chaîne Python `", '"` contient une virgule, une espace et **une apostrophe** — c'est cette apostrophe qui ouvre le texte SQL `'AD'`.

⚠️ **Le piège de l'apostrophe** : pour *Côte d'Ivoire*, on obtiendrait `'Côte d'Ivoire'`, refusé par H2. Il faut doubler l'apostrophe (partie 5) avant de construire la ligne, avec une petite fonction qui parcourt le texte caractère par caractère :
```python
def echappe(texte):
    resultat = ""
    for caractere in texte:
        if caractere == "'":
            resultat = resultat + "''"
        else:
            resultat = resultat + caractere
    return resultat
```

**Avec ces trois outils**, le programme de l'exercice 1 : ouvrir le CSV en lecture et `countries.sql` en écriture ; pour chaque pays, construire la ligne `INSERT` (avec `echappe` sur le nom) et l'écrire suivie de `"\n"` ; fermer les deux fichiers.

---
# 7. Interroger une table : `SELECT … FROM … WHERE …`

On ajoute une deuxième table pour avoir des exemples plus riches : quelques aéroports (comme dans l'exercice 2).

*On crée et on remplit la table `Airports` :*

```sql
CREATE TABLE Airports (
    id INT, code CHAR(4), name VARCHAR, type VARCHAR,
    latitude REAL, longitude REAL, altitude INT, country CHAR(2), city VARCHAR );
INSERT INTO Airports VALUES (101, 'LFPG', 'Charles de Gaulle International Airport', 'large_airport', 49.0128, 2.55, 392, 'FR', 'Paris');
INSERT INTO Airports VALUES (102, 'LFPO', 'Paris-Orly Airport', 'large_airport', 48.7233, 2.3794, 291, 'FR', 'Paris');
INSERT INTO Airports VALUES (103, 'LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.6291, 1.3638, 499, 'FR', 'Toulouse');
INSERT INTO Airports VALUES (104, 'LFPI', 'Paris Issy-les-Moulineaux Heliport', 'heliport', 48.8333, 2.2711, 112, 'FR', 'Paris');
INSERT INTO Airports VALUES (105, 'LEMD', 'Adolfo Suárez Madrid–Barajas Airport', 'large_airport', 40.4719, -3.5626, 1998, 'ES', 'Madrid');
INSERT INTO Airports VALUES (106, 'SBGR', 'Guarulhos International Airport', 'large_airport', -23.4356, -46.4731, 2459, 'BR', 'São Paulo');
INSERT INTO Airports VALUES (107, 'RJTT', 'Tokyo Haneda International Airport', 'large_airport', 35.5523, 139.78, 35, 'JP', 'Tokyo');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Airports ( id INT, code CHAR(4), name VARCHAR, type VARCHAR, latitude REAL, longitude REAL, altitude INT, country CHAR(2), city VARCHAR );
(Update count: 0)

INSERT INTO Airports VALUES (101, 'LFPG', 'Charles de Gaulle International Airport', 'large_airport', 49.0128, 2.55, 392, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES (102, 'LFPO', 'Paris-Orly Airport', 'large_airport', 48.7233, 2.3794, 291, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES (103, 'LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.6291, 1.3638, 499, 'FR', 'Toulouse');
(Update count: 1)

INSERT INTO Airports VALUES (104, 'LFPI', 'Paris Issy-les-Moulineaux Heliport', 'heliport', 48.8333, 2.2711, 112, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES (105, 'LEMD', 'Adolfo Suárez Madrid–Barajas Airport', 'large_airport', 40.4719, -3.5626, 1998, 'ES', 'Madrid');
(Update count: 1)

INSERT INTO Airports VALUES (106, 'SBGR', 'Guarulhos International Airport', 'large_airport', -23.4356, -46.4731, 2459, 'BR', 'São Paulo');
(Update count: 1)

INSERT INTO Airports VALUES (107, 'RJTT', 'Tokyo Haneda International Airport', 'large_airport', 35.5523, 139.78, 35, 'JP', 'Tokyo');
(Update count: 1)
```

### `SELECT` : choisir les colonnes

`SELECT colonnes FROM table;` affiche les colonnes demandées, pour **toutes** les lignes. `*` veut dire « toutes les colonnes ».

*Dans la console H2 :*

```sql
SELECT name, city FROM Airports;
```

*Résultat dans la console H2 :*

```text
NAME                                    | CITY
Charles de Gaulle International Airport | Paris
Paris-Orly Airport                      | Paris
Toulouse-Blagnac Airport                | Toulouse
Paris Issy-les-Moulineaux Heliport      | Paris
Adolfo Suárez Madrid–Barajas Airport    | Madrid
Guarulhos International Airport         | São Paulo
Tokyo Haneda International Airport      | Tokyo
(7 rows)
```

### `WHERE` : choisir les lignes

`WHERE condition` ne garde que les lignes qui vérifient la condition. Comparaisons : `=`, `<>` (différent), `<`, `>`, `<=`, `>=`. On combine avec `AND` (les deux), `OR` (l'une ou l'autre), `NOT`.

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE city = 'Paris' AND type = 'large_airport';
```

*Résultat dans la console H2 :*

```text
NAME
Charles de Gaulle International Airport
Paris-Orly Airport
(2 rows)
```

*Dans la console H2 :*

```sql
SELECT code, altitude FROM Airports WHERE altitude > 1000 OR country = 'JP';
```

*Résultat dans la console H2 :*

```text
CODE | ALTITUDE
LEMD | 1998
SBGR | 2459
RJTT | 35
(3 rows)
```

**Lire une requête dans le bon ordre :** on regarde d'abord `FROM` (dans quelle table ?), puis `WHERE` (quelles lignes ?), et enfin `SELECT` (quelles colonnes afficher ?).

### `LIKE` : chercher un mot dans un texte

`LIKE` compare avec un **motif** : `%` remplace n'importe quelle suite de caractères (même vide).

| Motif | Signification |
|:---|:---|
| `name LIKE 'Paris%'` | le nom **commence** par Paris |
| `name LIKE '%Airport'` | le nom **finit** par Airport |
| `name LIKE '%Orly%'` | le nom **contient** Orly |

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE name LIKE '%Paris%';
```

*Résultat dans la console H2 :*

```text
NAME
Paris-Orly Airport
Paris Issy-les-Moulineaux Heliport
(2 rows)
```

⚠️ Dans H2, `LIKE` **distingue les majuscules et les minuscules** : `'%paris%'` ne trouverait rien ici.

### `ORDER BY` : trier, `DISTINCT` : enlever les doublons

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR' ORDER BY altitude DESC;
```

*Résultat dans la console H2 :*

```text
NAME                                    | ALTITUDE
Toulouse-Blagnac Airport                | 499
Charles de Gaulle International Airport | 392
Paris-Orly Airport                      | 291
Paris Issy-les-Moulineaux Heliport      | 112
(4 rows)
```

`ORDER BY colonne` trie par ordre croissant (`ASC`, par défaut) ; `DESC` pour l'ordre décroissant. Le texte est trié par ordre alphabétique.

*Dans la console H2 :*

```sql
SELECT DISTINCT city FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
CITY
Paris
Toulouse
(2 rows)
```

Sans `DISTINCT`, Paris apparaîtrait trois fois (une fois par aéroport).

---
# 8. Compter, moyenne, minimum, maximum (fonctions d'agrégation)

Ces fonctions résument **toutes les lignes gardées** en **une seule valeur** :

| Fonction | Résultat |
|:---|:---|
| `COUNT(*)` | le nombre de lignes |
| `SUM(colonne)` | la somme |
| `AVG(colonne)` | la moyenne |
| `MIN(colonne)` / `MAX(colonne)` | la plus petite / la plus grande valeur |

*Dans la console H2 :*

```sql
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
SELECT AVG(altitude) FROM Airports;
SELECT MIN(latitude), MAX(altitude) FROM Airports;
```

*Résultat dans la console H2 :*

```text
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
COUNT(*)
4
(1 row)

SELECT AVG(altitude) FROM Airports;
AVG(ALTITUDE)
826.5714285714286
(1 row)

SELECT MIN(latitude), MAX(altitude) FROM Airports;
MIN(LATITUDE) | MAX(ALTITUDE)
-23.4356      | 2459
(1 row)
```

- Le `WHERE` agit **d'abord** : `COUNT(*)` compte les aéroports **français** (4).
- ⚠️ `SELECT name, MIN(latitude) FROM Airports;` est **refusé** : `MIN` donne une seule valeur, alors que `name` en a une par ligne. Pour avoir **le nom** de l'aéroport le plus au sud, deux méthodes :

*Méthode 1 : trier, et lire la première ligne du résultat.*

```sql
SELECT name, latitude FROM Airports ORDER BY latitude;
```

*Résultat dans la console H2 :*

```text
NAME                                    | LATITUDE
Guarulhos International Airport         | -23.4356
Tokyo Haneda International Airport      | 35.5523
Adolfo Suárez Madrid–Barajas Airport    | 40.4719
Toulouse-Blagnac Airport                | 43.6291
Paris-Orly Airport                      | 48.7233
Paris Issy-les-Moulineaux Heliport      | 48.8333
Charles de Gaulle International Airport | 49.0128
(7 rows)
```

*Méthode 2 : une requête dans la requête. H2 calcule d'abord la parenthèse (la latitude minimale), puis cherche l'aéroport qui a cette latitude.*

```sql
SELECT name FROM Airports WHERE latitude = (SELECT MIN(latitude) FROM Airports);
```

*Résultat dans la console H2 :*

```text
NAME
Guarulhos International Airport
(1 row)
```

La méthode 2 répond en **une seule requête** (ce que demandent les exercices 2 et 3 : « le nom de la station située la plus au sud », « la station avec le plus grand nombre de vélos »).

> Le programme dit qu'on n'utilise **pas** `GROUP BY` (« pour chaque continent… ») : si une question semble le demander, on fait une requête par cas.

---
# 9. Clé primaire

**Problème :** rien n'empêche d'insérer deux fois la France dans notre table. Comment distinguer deux lignes ?

**Définition (polycopié).** Un **identifiant** (ou **clé**) d'une relation est un attribut, ou un groupe d'attributs, qui permet d'identifier **de manière unique** un enregistrement : deux lignes ne peuvent pas avoir le même identifiant. Il y a parfois plusieurs choix possibles (les **clés candidates**) ; celui qu'on choisit s'appelle la **clé primaire**.

- Pour savoir si un attribut peut être une clé, on se demande : **deux objets différents peuvent-ils avoir la même valeur ?** Si oui, ce n'est pas une clé. Deux pays n'ont jamais le même code ISO ; deux élèves peuvent avoir le même nom.
- C'est pourquoi on **invente** souvent un numéro (`id`, matricule, numéro de commande…).
- Dans un schéma, la clé primaire est **soulignée** : Countries(<u>id</u>, code, name, continent).

On la déclare avec `PRIMARY KEY`, et le SGBD **refuse** les doublons (la *contrainte de relation*, ou contrainte de clé primaire) :

*Dans la console H2 :*

```sql
CREATE TABLE Pays (
    code CHAR(2), name VARCHAR,
    PRIMARY KEY (code) );
INSERT INTO Pays VALUES ('FR', 'France');
INSERT INTO Pays VALUES ('FR', 'France');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Pays ( code CHAR(2), name VARCHAR, PRIMARY KEY (code) );
(Update count: 0)

INSERT INTO Pays VALUES ('FR', 'France');
(Update count: 1)

INSERT INTO Pays VALUES ('FR', 'France');
Erreur : Violation d'index unique ou clé primaire: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"
Unique index or primary key violation: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"; SQL statement:
INSERT INTO Pays VALUES ('FR', 'France') [23505-252]
```

Pour une clé faite de **plusieurs** attributs, on les met tous dans la même parenthèse : `PRIMARY KEY (aid, cid, dateachat)` (exercice 6).

---
# 10. Plusieurs tables : clé étrangère et jointure

Dans `Airports`, la colonne `country` contient un code de pays (`'FR'`) : elle **fait référence** à un pays de la table `Countries`.

**Définition (polycopié).** Un attribut qui fait référence à la clé primaire d'une autre relation est une **clé étrangère**. Dans un schéma, on la marque d'un astérisque (ou d'un #) :

Airports(<u>id</u>, code, name, type, latitude, longitude, altitude, country\*, city) — `country` fait référence à `Countries.code`.

### La jointure : `JOIN … ON …`

**Problème :** on veut le **nom du pays** de chaque aéroport. Le nom de l'aéroport est dans `Airports`, le nom du pays dans `Countries`. La **jointure** recolle les deux tables : chaque ligne de `Airports` est mise à côté de la ligne de `Countries` qui a le même code de pays.

*Dans la console H2 :*

```sql
SELECT Airports.name, Countries.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code;
```

*Résultat dans la console H2 :*

```text
NAME                                    | NAME
Charles de Gaulle International Airport | France
Paris-Orly Airport                      | France
Toulouse-Blagnac Airport                | France
Paris Issy-les-Moulineaux Heliport      | France
Adolfo Suárez Madrid–Barajas Airport    | Spain
Guarulhos International Airport         | Brazil
Tokyo Haneda International Airport      | Japan
(7 rows)
```

- `ON` dit **comment** recoller : la clé étrangère `Airports.country` doit être égale à la clé primaire `Countries.code`.
- Quand une colonne existe dans les deux tables (ici `name`), on la **préfixe** par le nom de la table : `Airports.name`, `Countries.name`. Sinon H2 répond que la colonne est ambiguë.
- Après la jointure, on utilise `WHERE`, `ORDER BY`, `COUNT`… comme avec une seule table :

*Les aéroports d'Espagne, en partant du nom du pays :*

```sql
SELECT Airports.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code
WHERE Countries.name = 'Spain';
```

*Résultat dans la console H2 :*

```text
NAME
Adolfo Suárez Madrid–Barajas Airport
(1 row)
```

### La contrainte d'intégrité référentielle

Si on **déclare** la clé étrangère avec `FOREIGN KEY … REFERENCES …`, le SGBD vérifie que chaque valeur existe bien dans l'autre table :

*Dans la console H2 :*

```sql
CREATE TABLE Villes (
    nom VARCHAR, pays CHAR(2),
    PRIMARY KEY (nom),
    FOREIGN KEY (pays) REFERENCES Pays(code) );
INSERT INTO Villes VALUES ('Lyon', 'FR');
INSERT INTO Villes VALUES ('Rome', 'IT');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Villes ( nom VARCHAR, pays CHAR(2), PRIMARY KEY (nom), FOREIGN KEY (pays) REFERENCES Pays(code) );
(Update count: 0)

INSERT INTO Villes VALUES ('Lyon', 'FR');
(Update count: 1)

INSERT INTO Villes VALUES ('Rome', 'IT');
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"
Referential integrity constraint violation: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"; SQL statement:
INSERT INTO Villes VALUES ('Rome', 'IT') [23506-252]
```

- Lyon est acceptée (le pays `FR` existe dans `Pays`), Rome est **refusée** : le pays `IT` n'existe pas.
- **Conséquence sur l'ordre** : on crée et on remplit d'abord la table **référencée** (`Pays`), puis celle qui y fait référence (`Villes`). Pour supprimer, c'est l'inverse : on ne peut pas supprimer un pays qui a encore des villes.

**Les trois contraintes d'intégrité** vérifiées par le SGBD (programme officiel) :

| Contrainte | Ce qui est vérifié | Exemple refusé |
|:---|:---|:---|
| de **domaine** | le type de chaque valeur | `' IM '` dans un `CHAR(2)` |
| de **relation** (clé primaire) | pas deux lignes avec la même clé | la France insérée deux fois |
| de **référence** (clé étrangère) | la valeur référencée existe | une ville dans le pays `IT` inexistant |

---
# 11. Pourquoi plusieurs tables ? (les anomalies)

Le polycopié montre un seul grand tableau élève / classe / matière / professeur. Le nom de chaque professeur y est **répété** sur des dizaines de lignes : c'est une **redondance**. Elle provoque des **anomalies** :
- **de mise à jour** : si un professeur change, il faut modifier toutes ses lignes ; en oublier une rend la base incohérente ;
- **d'insertion** : impossible d'enregistrer un nouveau professeur tant qu'il n'a pas d'élève ;
- **de suppression** : si on supprime le dernier élève d'une matière, on perd aussi le nom de son professeur.

**La solution :** découper en plusieurs tables (une table des élèves, une des matières avec leur professeur…), reliées par des **clés étrangères**, puis les recoller avec `JOIN` quand on en a besoin. Chaque information n'est écrite **qu'une seule fois**.

---
# 12. Modifier et supprimer : `UPDATE`, `DELETE`

`UPDATE table SET colonne = valeur WHERE condition;` modifie les lignes qui vérifient la condition.

*Dans la console H2 :*

```sql
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
SELECT code, altitude FROM Airports WHERE city = 'Paris';
```

*Résultat dans la console H2 :*

```text
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
(Update count: 1)

SELECT code, altitude FROM Airports WHERE city = 'Paris';
CODE | ALTITUDE
LFPG | 392
LFPO | 300
LFPI | 112
(3 rows)
```

`DELETE FROM table WHERE condition;` supprime les lignes qui vérifient la condition.

*Dans la console H2 :*

```sql
DELETE FROM Airports WHERE type = 'heliport';
SELECT COUNT(*) FROM Airports;
```

*Résultat dans la console H2 :*

```text
DELETE FROM Airports WHERE type = 'heliport';
(Update count: 1)

SELECT COUNT(*) FROM Airports;
COUNT(*)
6
(1 row)
```

⚠️ **Ne jamais oublier le `WHERE`** : `UPDATE Airports SET altitude = 300;` modifierait **tous** les aéroports, et `DELETE FROM Airports;` les supprimerait **tous**. Il n'y a pas de bouton « annuler ».

---
# 13. La méthode pour écrire une requête

Devant une question (« Quel est le code ICAO de l'aéroport de Toulouse-Blagnac ? »), on répond à **quatre questions, dans cet ordre** :

| Étape | Question à se poser | Exemple | Ce qu'on écrit |
|:---|:---|:---|:---|
| 1 | Quelles **colonnes** veut-on afficher ? | le code ICAO | `SELECT code` |
| 2 | Dans quelle(s) **table(s)** sont-elles ? Et les colonnes de la condition ? | `Airports` | `FROM Airports` (+ `JOIN … ON …` s'il en faut deux) |
| 3 | Quelles **lignes** garde-t-on ? | celle de Toulouse-Blagnac | `WHERE name LIKE '%Toulouse-Blagnac%'` |
| 4 | Faut-il **compter, trier, enlever les doublons** ? | non | (`COUNT`, `ORDER BY`, `DISTINCT`) |

**Conseils :**
- regarder d'abord le contenu d'une table avec `SELECT * FROM table;` pour voir comment les valeurs sont écrites (`'heliport'` ? `'FR'` ou `'France'` ?) ;
- construire la requête **petit à petit** : d'abord sans le `WHERE`, puis l'ajouter, puis la jointure ;
- « combien… » → `COUNT` ; « le plus… » → `MAX` / `MIN` ou `ORDER BY` ; « quels sont les … différents » → `DISTINCT`.

## Fiche récapitulative

```sql
-- créer, remplir
CREATE TABLE T (a INT, b VARCHAR, PRIMARY KEY (a), FOREIGN KEY (b) REFERENCES U(c));
INSERT INTO T (a, b) VALUES (1, 'texte');
RUNSCRIPT FROM 'chemin/vers/fichier.sql';

-- interroger
SELECT a, b FROM T WHERE a > 3 AND b LIKE '%mot%' ORDER BY a DESC;
SELECT DISTINCT b FROM T;
SELECT COUNT(*), SUM(a), AVG(a), MIN(a), MAX(a) FROM T WHERE ...;
SELECT b FROM T WHERE a = (SELECT MAX(a) FROM T);

-- plusieurs tables
SELECT T.a, U.d FROM T JOIN U ON T.b = U.c WHERE ...;

-- modifier, supprimer (toujours avec WHERE !)
UPDATE T SET b = 'nouveau' WHERE a = 1;
DELETE FROM T WHERE a = 1;
```

| Mot | Sens |
|:---|:---|
| relation / attribut / enregistrement | table / colonne / ligne |
| domaine | type des valeurs d'un attribut |
| schéma | liste des attributs et de leurs types, clé primaire soulignée, clés étrangères marquées |
| clé primaire | identifie chaque ligne de façon unique |
| clé étrangère | fait référence à la clé primaire d'une autre table |
| SGBD | logiciel qui gère la base : persistance, accès concurrents, efficacité, sécurité |

> ⚠️ **Liberté prise :** les données des exemples (quelques pays et aéroports) sont une petite sélection faite pour ce cours ; les vraies tables de l'exercice 2 sont beaucoup plus grandes et ont d'autres identifiants. Les résultats ont été obtenus avec H2 et recopiés (sans la durée d'exécution, et avec « Erreur : » à la place du nom technique de l'erreur Java).